## Setup and Imports

This notebook starts with the package imports and shared constants needed by all sections. If a package is missing, install it before running the notebook.

In [45]:
# Core utilities shared by every section in the notebook.
import json
import os
import re
import shutil
from pathlib import Path
from difflib import SequenceMatcher, get_close_matches

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

# Word2Vec and fastText are optional; TF-IDF still runs if the embedding libraries are missing.
try:
    import gensim.downloader as api
    GENSIM_AVAILABLE = True
except ImportError:
    api = None
    GENSIM_AVAILABLE = False

Word2Vec = None
FastText = None

try:
    from compress_fasttext import models as fasttext_models
    COMPRESS_FASTTEXT_AVAILABLE = True
except ImportError:
    fasttext_models = None
    COMPRESS_FASTTEXT_AVAILABLE = False

DATA_PATH = Path("25k IMDb movie Dataset.csv")
OUTPUT_DIR = Path("artifacts")
GENSIM_DATA_DIR = Path("gensim_data")
GENSIM_DATA_DIR.mkdir(exist_ok=True)
os.environ["GENSIM_DATA_DIR"] = str(GENSIM_DATA_DIR)

TOP_N = 10
SAMPLE_QUERY = "avatar"
RANDOM_STATE = 42
WORD2VEC_MODEL_NAME = "glove-wiki-gigaword-50"
FASTTEXT_MODEL_PATH = GENSIM_DATA_DIR / "cc.en.300.compressed.bin"

# GloVe is a compact pretrained baseline (~66 MB) for quick CPU testing.

# fastText is a larger pretrained subword model (~21-25 MB compressed) for OOV-friendly matching.
if not COMPRESS_FASTTEXT_AVAILABLE:
    print("compress_fasttext is not installed; fastText section will be skipped until it is available.")


## TF-IDF Main Flow

This is the primary recommender path in the notebook. The Word2Vec and fastText blocks later use the same cleaned movie table and the same title-matching helper, but they keep their own saved artifacts.

In [55]:
# TF-IDF gives one sparse row per movie; the matrix stays compact because most entries are zero.
tfidf_query = SAMPLE_QUERY
tfidf_match, tfidf_match_score, tfidf_suggestions = find_best_title(tfidf_query, movie_titles)

if tfidf_match is None:
    tfidf_recommendations = []
else:
    tfidf_index = movies.index[movies["movie title"] == tfidf_match][0]
    tfidf_recommendations = recommend_from_index(tfidf_index, tfidf_nn, tfidf_features, movies, top_n=TOP_N)

# Reuse the same display format so the comparison sections are easy to scan.
print_recommendations(
    "TF-IDF",
    tfidf_query,
    tfidf_recommendations,
    matched_title=tfidf_match,
    score=tfidf_match_score,
    suggestions=tfidf_suggestions,
 )

pd.DataFrame(tfidf_recommendations)


[TF-IDF] query: avatar
matched: Avatar (1.00)
1 . Avatar 5 | score: 0.7132
2 . Avatar 4 | score: 0.6517
3 . Avatar 3 | score: 0.5237
4 . Avatar: The Way of Water | score: 0.5117
5 . The Abyss | score: 0.4942
6 . Terminator 2: Judgment Day | score: 0.3955
7 . Aliens | score: 0.2883
8 . The Terminator | score: 0.2407
9 . Alita: Battle Angel | score: 0.1481
10 . I Love My Dad | score: 0.1276


,movie title,score
0,Avatar 5,0.7132
1,Avatar 4,0.6517
2,Avatar 3,0.5237
3,Avatar: The Way of Water,0.5117
4,The Abyss,0.4942
5,Terminator 2: Judgment Day,0.3955
6,Aliens,0.2883
7,The Terminator,0.2407
8,Alita: Battle Angel,0.1481
9,I Love My Dad,0.1276


## Word2Vec Comparison

This section uses the same cleaned movie text but trains a Word2Vec model and builds separate recommendation artifacts for comparison against TF-IDF.

In [ ]:
# Word2Vec is a dense embedding baseline; it summarizes the movie text into compact vectors.
os.environ["GENSIM_DATA_DIR"] = str(GENSIM_DATA_DIR)
word2vec_tokens = movies["features"].apply(tokenize_text).tolist()

try:
    from gensim.models import Word2Vec

    word2vec_model = Word2Vec(
        sentences=word2vec_tokens,
        vector_size=100,
        window=5,
        min_count=1,
        workers=2,
        epochs=20,
        seed=RANDOM_STATE,
    )
    GENSIM_AVAILABLE = True
except Exception as exc:
    Word2Vec = None
    word2vec_model = None
    GENSIM_AVAILABLE = False
    print(f"Word2Vec is unavailable; skipping this section. Reason: {exc}")
    word2vec_tokens = []
    word2vec_vectors = None
    word2vec_nn = None
else:
    word2vec_vectors = build_embedding_matrix(word2vec_model, word2vec_tokens, word2vec_model.vector_size)
    word2vec_nn = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1)
    word2vec_nn.fit(word2vec_vectors)
    print("Word2Vec model trained from the movie corpus.")


Word2Vec model trained from the movie corpus.


In [ ]:
if GENSIM_AVAILABLE:
    word2vec_query = SAMPLE_QUERY
    word2vec_match, word2vec_match_score, word2vec_suggestions = find_best_title(word2vec_query, movie_titles)

    if word2vec_match is None:
        word2vec_recommendations = []
    else:
        word2vec_index = movies.index[movies["movie title"] == word2vec_match][0]
        word2vec_recommendations = recommend_from_index(word2vec_index, word2vec_nn, word2vec_vectors, movies, top_n=TOP_N)

    print_recommendations(
        "Word2Vec",
        word2vec_query,
        word2vec_recommendations,
        matched_title=word2vec_match,
        score=word2vec_match_score,
        suggestions=word2vec_suggestions,
    )

    pd.DataFrame(word2vec_recommendations)
else:
    print("Skipping Word2Vec preview because gensim is unavailable.")


[Word2Vec] query: avatar
matched: Avatar (1.00)
1 . Avatar: The Way of Water | score: 0.9095
2 . Logan | score: 0.8749
3 . CarGo | score: 0.8748
4 . The Abyss | score: 0.8701
5 . You Only Live Twice | score: 0.8684
6 . Gardens of Stone | score: 0.8639
7 . Free Willy 2: The Adventure Home | score: 0.8586
8 . Night Passage | score: 0.8585
9 . Maze Runner: The Death Cure | score: 0.8577
10 . Aliens | score: 0.8576


In [ ]:
os.environ["GENSIM_DATA_DIR"] = str(GENSIM_DATA_DIR)
try:
    from gensim.models import FastText
    FastText = FastText
    GENSIM_AVAILABLE = True
except ImportError:
    FastText = None
    GENSIM_AVAILABLE = False
    print("fastText is unavailable; skipping this section.")

if GENSIM_AVAILABLE:
    FASTTEXT_MODEL_PATH.parent.mkdir(exist_ok=True)
    fasttext_tokens = movies["features"].apply(tokenize_text).tolist()
    fasttext_model = FastText(
        sentences=fasttext_tokens,
        vector_size=100,
        window=5,
        min_count=1,
        workers=2,
        epochs=20,
        seed=RANDOM_STATE,
    )

    fasttext_vectors = build_embedding_matrix(fasttext_model, fasttext_tokens, fasttext_model.vector_size)
    fasttext_nn = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1)
    fasttext_nn.fit(fasttext_vectors)
    print("FastText model trained from the movie corpus.")
else:
    fasttext_tokens = []
    fasttext_model = None
    fasttext_nn = None
    fasttext_vectors = None
    print("Skipping fastText section because gensim is unavailable.")


FastText model trained from the movie corpus.


In [22]:
if GENSIM_AVAILABLE:
    fasttext_query = SAMPLE_QUERY
    fasttext_match, fasttext_match_score, fasttext_suggestions = find_best_title(fasttext_query, movie_titles)

    if fasttext_match is None:
        fasttext_recommendations = []
    else:
        fasttext_index = movies.index[movies["movie title"] == fasttext_match][0]
        fasttext_recommendations = recommend_from_index(fasttext_index, fasttext_nn, fasttext_vectors, movies, top_n=TOP_N)

    print_recommendations(
        "fastText",
        fasttext_query,
        fasttext_recommendations,
        matched_title=fasttext_match,
        score=fasttext_match_score,
        suggestions=fasttext_suggestions,
    )

    pd.DataFrame(fasttext_recommendations)
else:
    print("Skipping fastText preview because gensim is unavailable.")


[fastText] query: avatar
matched: Avatar (1.00)
1 . God of Thunder | score: 0.9415
2 . The Scorpion King | score: 0.9401
3 . The Purge | score: 0.94
4 . The Cascadia Treasure | score: 0.9395
5 . Logan | score: 0.9383
6 . GoldenEye | score: 0.9375
7 . Oz the Great and Powerful | score: 0.9374
8 . Barbie: Star Light Adventure | score: 0.9372
9 . Mighty Morphin Power Rangers: The Movie | score: 0.9366
10 . Primeval | score: 0.9365


In [ ]:
try:
    from sentence_transformers import SentenceTransformer

    ALL_MINILM_MODEL_NAME = "all-MiniLM-L6-v2"
    ALL_MINILM_AVAILABLE = True
except Exception as exc:
    SentenceTransformer = None
    ALL_MINILM_MODEL_NAME = "all-MiniLM-L6-v2"
    ALL_MINILM_AVAILABLE = False
    print(f"all-MiniLM-L6-v2 is unavailable; skipping this section. Reason: {exc}")

if ALL_MINILM_AVAILABLE:
    all_minilm_model = SentenceTransformer(ALL_MINILM_MODEL_NAME)
    all_minilm_texts = movies["features"].fillna("").astype(str).tolist()
    all_minilm_vectors = all_minilm_model.encode(all_minilm_texts, convert_to_numpy=True, show_progress_bar=False)
    all_minilm_nn = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1)
    all_minilm_nn.fit(all_minilm_vectors)
    print("all-MiniLM-L6-v2 embeddings built from the movie corpus.")
else:
    all_minilm_model = None
    all_minilm_texts = []
    all_minilm_vectors = None
    all_minilm_nn = None
    print("Skipping all-MiniLM-L6-v2 comparison because sentence-transformers is unavailable.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1210.65it/s]


all-MiniLM-L6-v2 embeddings built from the movie corpus.


In [ ]:
if ALL_MINILM_AVAILABLE:
    all_minilm_query = SAMPLE_QUERY
    all_minilm_match, all_minilm_match_score, all_minilm_suggestions = find_best_title(all_minilm_query, movie_titles)

    if all_minilm_match is None:
        all_minilm_recommendations = []
    else:
        all_minilm_index = movies.index[movies["movie title"] == all_minilm_match][0]
        all_minilm_recommendations = recommend_from_index(all_minilm_index, all_minilm_nn, all_minilm_vectors, movies, top_n=TOP_N)

    print_recommendations(
        "all-MiniLM-L6-v2",
        all_minilm_query,
        all_minilm_recommendations,
        matched_title=all_minilm_match,
        score=all_minilm_match_score,
        suggestions=all_minilm_suggestions,
    )

    pd.DataFrame(all_minilm_recommendations)
else:
    print("Skipping all-MiniLM-L6-v2 preview because sentence-transformers is unavailable.")


[all-MiniLM-L6-v2] query: avatar
matched: Avatar (1.00)
1 . Avatar: The Way of Water | score: 0.7048
2 . Avatar 3 | score: 0.6211
3 . Avatar 4 | score: 0.6065
4 . Rebel Moon | score: 0.574
5 . Avatar 5 | score: 0.5549
6 . Kong: Skull Island | score: 0.5388
7 . Monster Hunter | score: 0.5246
8 . IO | score: 0.5241
9 . Moonbound | score: 0.5213
10 . Oblivion | score: 0.5208


In [ ]:
comparison_rows = [
    {
        "approach": "TF-IDF",
        "main_role": "main flow",
        "saved_artifacts": "artifacts/tfidf_recommender.joblib, artifacts/tfidf_features.npz, artifacts/tfidf_metadata.json",
        "notes": "baseline content-based recommender",
    },
    {
        "approach": "Word2Vec",
        "main_role": "comparison",
        "saved_artifacts": "artifacts/word2vec_recommender.joblib, artifacts/word2vec_vectors.npy, artifacts/word2vec_metadata.json",
        "notes": "embedding average over trained word vectors",
    },
    {
        "approach": "fastText",
        "main_role": "comparison",
        "saved_artifacts": "artifacts/fasttext_recommender.joblib, artifacts/fasttext_vectors.npy, artifacts/fasttext_metadata.json",
        "notes": "subword-aware embedding comparison",
    },
    {
        "approach": "all-MiniLM-L6-v2",
        "main_role": "comparison",
        "saved_artifacts": "artifacts/all_minilm_recommender.joblib, artifacts/all_minilm_vectors.npy, artifacts/all_minilm_metadata.json",
        "notes": "lightweight transformer sentence embeddings",
    },
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,approach,main_role,saved_artifacts,notes
0,TF-IDF,main flow,"artifacts/tfidf_recommender.joblib, artifacts/...",baseline content-based recommender
1,Word2Vec,comparison,"artifacts/word2vec_recommender.joblib, artifac...",embedding average over trained word vectors
2,fastText,comparison,"artifacts/fasttext_recommender.joblib, artifac...",subword-aware embedding comparison
3,all-MiniLM-L6-v2,comparison,"artifacts/all_minilm_recommender.joblib, artif...",lightweight transformer sentence embeddings


## ⭐⭐⭐⭐☆ (4.2 / 5) — all-MiniLM-L6-v2 (List 4)

- Best semantic alignment: returns coherent sci-fi context and strong franchise continuity.
- Limitation: unweighted metadata still introduces some noise from long descriptive fields.

## ⭐⭐⭐⭐☆ (4.0 / 5) — 2nd Place: TF-IDF (List 1)

- Strong auteur signal: the model reliably surfaces James Cameron titles using `Director` and `Writer` overlap.
- Limitation: raw token weighting is still sensitive to long text fields and loses some semantic precision.

## ⭐⭐⭐☆☆ (2.0 / 5) — 3rd Place: Word2Vec (List 3)

- Good top hit, but unstable after the first result due to averaged embedding noise.
- Limitation: simple vector averaging dilutes semantic structure when mixed with broad metadata.

## ⭐☆☆☆☆ (1.0 / 5) — 4th Place: FastText (List 2)

- Poor ranking quality: noisy cluster with weak semantic separation.
- Limitation: subword matching over long unweighted text overwhelms the actual similarity signal.

## Pretrained Embedders (Compact GloVe + Mini FastText)

This dedicated section loads the compact pretrained embedding models, saves the mini FastText model into the project cache, and prepares the embedding-based recommender before the final weighted TF-IDF stage.

In [52]:
# Pretrained embedders: compact GloVe + Mini FastText.
# This section is kept separate and saved into the project cache before the final weighted TF-IDF stage.
import gensim.downloader as api
import compress_fasttext
from pathlib import Path

GENSIM_DATA_DIR = Path("gensim_data")
GENSIM_DATA_DIR.mkdir(exist_ok=True)

print("Loading GloVe...")
try:
    pretrained_word2vec_model = api.load("glove-wiki-gigaword-50")
    print("GloVe loaded successfully.")
except Exception as exc:
    pretrained_word2vec_model = None
    print(f"GloVe download failed: {exc}")

print("Loading Mini FastText...")
FASTTEXT_URL = "https://github.com/avidale/compress-fasttext/releases/download/v0.0.4/cc.en.300.compressed.bin"
FASTTEXT_LOCAL_PATH = GENSIM_DATA_DIR / "cc.en.300.compressed.bin"

try:
    if FASTTEXT_LOCAL_PATH.exists():
        pretrained_fasttext_model = compress_fasttext.models.CompressedFastTextKeyedVectors.load(str(FASTTEXT_LOCAL_PATH))
        print(f"Loaded Mini FastText from local cache: {FASTTEXT_LOCAL_PATH}")
    else:
        pretrained_fasttext_model = compress_fasttext.models.CompressedFastTextKeyedVectors.load(FASTTEXT_URL)
        try:
            pretrained_fasttext_model.save(str(FASTTEXT_LOCAL_PATH))
            print(f"Saved Mini FastText to {FASTTEXT_LOCAL_PATH}")
        except Exception as save_exc:
            print(f"Mini FastText loaded but could not be saved locally: {save_exc}")
    print("Mini FastText loaded successfully.")
except Exception as exc:
    pretrained_fasttext_model = None
    print(f"Mini FastText download failed: {exc}")

if pretrained_word2vec_model is not None and pretrained_fasttext_model is not None:
    print("Both small pretrained models are ready.")
    print(f"Mini FastText cache: {FASTTEXT_LOCAL_PATH}")
else:
    print("One or both small pretrained models are unavailable; the notebook will keep its trained baselines as fallback.")

if pretrained_word2vec_model is not None and pretrained_fasttext_model is not None:
    pretrained_word2vec_vectors = []
    pretrained_fasttext_vectors = []

    for tokens in movies["features"].apply(tokenize_text).tolist():
        glove_vectors = [pretrained_word2vec_model[token] for token in tokens if token in pretrained_word2vec_model]
        fasttext_vectors_for_movie = [pretrained_fasttext_model[token] for token in tokens if token in pretrained_fasttext_model]

        if glove_vectors:
            pretrained_word2vec_vectors.append(np.mean(glove_vectors, axis=0))
        else:
            pretrained_word2vec_vectors.append(np.zeros(pretrained_word2vec_model.vector_size, dtype=np.float32))

        if fasttext_vectors_for_movie:
            pretrained_fasttext_vectors.append(np.mean(fasttext_vectors_for_movie, axis=0))
        else:
            pretrained_fasttext_vectors.append(np.zeros(pretrained_fasttext_model.vector_size, dtype=np.float32))

    pretrained_word2vec_vectors = np.vstack(pretrained_word2vec_vectors)
    pretrained_fasttext_vectors = np.vstack(pretrained_fasttext_vectors)

    pretrained_word2vec_nn = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1)
    pretrained_word2vec_nn.fit(pretrained_word2vec_vectors)

    pretrained_fasttext_nn = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1)
    pretrained_fasttext_nn.fit(pretrained_fasttext_vectors)

    print("Pretrained embedding recommender built from GloVe and Mini FastText embeddings.")
else:
    pretrained_word2vec_vectors = None
    pretrained_fasttext_vectors = None
    pretrained_word2vec_nn = None
    pretrained_fasttext_nn = None

Loading GloVe...
GloVe loaded successfully.
Loading Mini FastText...
Loaded Mini FastText from local cache: gensim_data\cc.en.300.compressed.bin
Mini FastText loaded successfully.
Both small pretrained models are ready.
Mini FastText cache: gensim_data\cc.en.300.compressed.bin
Pretrained embedding recommender built from GloVe and Mini FastText embeddings.


In [53]:
# Example input-based recommendation using the pretrained embedding stage.
# This keeps the pretrained section simple and lets you test one real movie query immediately.
pretrained_query = "avatar"

if pretrained_word2vec_nn is not None and pretrained_word2vec_vectors is not None:
    pretrained_match, pretrained_match_score, pretrained_suggestions = find_best_title(pretrained_query, movie_titles)

    if pretrained_match is None:
        pretrained_recommendations = []
    else:
        pretrained_index = movies.index[movies["movie title"] == pretrained_match][0]
        pretrained_recommendations = recommend_from_index(
            pretrained_index,
            pretrained_word2vec_nn,
            pretrained_word2vec_vectors,
            movies,
            top_n=TOP_N,
        )

    print_recommendations(
        "Pretrained GloVe",
        pretrained_query,
        pretrained_recommendations,
        matched_title=pretrained_match,
        score=pretrained_match_score,
        suggestions=pretrained_suggestions,
    )

    pd.DataFrame(pretrained_recommendations)
else:
    print("Pretrained embedding recommender is not available yet; run the GloVe/Mini FastText setup first.")

# Example input-based recommendation using the mini pretrained FastText embedder.
if pretrained_fasttext_nn is not None and pretrained_fasttext_vectors is not None:
    pretrained_fasttext_match, pretrained_fasttext_match_score, pretrained_fasttext_suggestions = find_best_title(pretrained_query, movie_titles)

    if pretrained_fasttext_match is None:
        pretrained_fasttext_recommendations = []
    else:
        pretrained_fasttext_index = movies.index[movies["movie title"] == pretrained_fasttext_match][0]
        pretrained_fasttext_recommendations = recommend_from_index(
            pretrained_fasttext_index,
            pretrained_fasttext_nn,
            pretrained_fasttext_vectors,
            movies,
            top_n=TOP_N,
        )

    print_recommendations(
        "Pretrained Mini FastText",
        pretrained_query,
        pretrained_fasttext_recommendations,
        matched_title=pretrained_fasttext_match,
        score=pretrained_fasttext_match_score,
        suggestions=pretrained_fasttext_suggestions,
    )

    pd.DataFrame(pretrained_fasttext_recommendations)
else:
    print("Pretrained Mini FastText recommender is not available yet; run the GloVe/Mini FastText setup first.")



[Pretrained GloVe] query: avatar
matched: Avatar (1.00)
1 . Avatar: The Way of Water | score: 0.9773
2 . Aquaman | score: 0.9769
3 . Once Upon a Time in the Apocalypse | score: 0.9735
4 . Baby: Secret of the Lost Legend | score: 0.9716
5 . The Lord Protector | score: 0.9709
6 . Donovan's Reef | score: 0.9689
7 . Charlie's Ghost Story | score: 0.9688
8 . It Comes at Night | score: 0.9686
9 . Clarence, the Cross-Eyed Lion | score: 0.9677
10 . Return to Nim's Island | score: 0.9676

[Pretrained Mini FastText] query: avatar
matched: Avatar (1.00)
1 . The Northlander | score: 0.9416
2 . Wicker Park | score: 0.939
3 . Between Waves | score: 0.9386
4 . The Cut | score: 0.9375
5 . Zorro: The Gay Blade | score: 0.937
6 . Burden | score: 0.9369
7 . Mumbaki | score: 0.9369
8 . The Martian | score: 0.9369
9 . Finding Nemo | score: 0.9366
10 . The Ultimate Weapon | score: 0.9348


## Weighted Field Recommendation (Final Optimized Section)

This final section implements the field-level weighting strategy you described: title, director, and keyword fields are weighted more heavily than generic overview text so the recommender behaves more like the optimized List 1 version.

In [54]:
field_weights = {
    "movie title": 4.0,
    "Director": 3.0,
    "Generes": 2.0,
    "Plot Kyeword": 2.5,
    "Top 5 Casts": 1.5,
    "Writer": 1.0,
    "Overview":  0.5,
}

# Build a weighted text string per movie so the TF-IDF model emphasizes important metadata.
def build_weighted_features(row):
    weighted_parts = []
    for column, weight in field_weights.items():
        value = str(row.get(column, "")).strip()
        if not value:
            continue
        repeats = max(1, int(round(weight)))
        weighted_parts.append((value + " ") * repeats)
    return " ".join(weighted_parts)

movies["weighted_features"] = movies[text_columns].apply(build_weighted_features, axis=1)

weighted_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
)
weighted_features = weighted_vectorizer.fit_transform(movies["weighted_features"])
weighted_nn = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1)
weighted_nn.fit(weighted_features)

print("Weighted field TF-IDF recommender built.")

Weighted field TF-IDF recommender built.


In [ ]:
weighted_query = SAMPLE_QUERY
weighted_match, weighted_match_score, weighted_suggestions = find_best_title(weighted_query, movie_titles)

if weighted_match is None:
    weighted_recommendations = []
else:
    weighted_index = movies.index[movies["movie title"] == weighted_match][0]
    weighted_recommendations = recommend_from_index(weighted_index, weighted_nn, weighted_features, movies, top_n=TOP_N)

print_recommendations(
    "Weighted Fields",
    weighted_query,
    weighted_recommendations,
    matched_title=weighted_match,
    score=weighted_match_score,
    suggestions=weighted_suggestions,
)

pd.DataFrame(weighted_recommendations)


[Weighted Fields] query: avatar
matched: Avatar (1.00)
1 . Avatar 5 | score: 0.8225
2 . Avatar 4 | score: 0.761
3 . Avatar 3 | score: 0.6746
4 . Avatar: The Way of Water | score: 0.6233
5 . The Abyss | score: 0.5231
6 . Terminator 2: Judgment Day | score: 0.4013
7 . Aliens | score: 0.3631
8 . The Terminator | score: 0.3054
9 . True Lies | score: 0.2586
10 . Avatar Purusha | score: 0.2482


,movie title,score
0,Avatar 5,0.8225
1,Avatar 4,0.7610
2,Avatar 3,0.6746
3,Avatar: The Way of Water,0.6233
4,The Abyss,0.5231
5,Terminator 2: Judgment Day,0.4013
6,Aliens,0.3631
7,The Terminator,0.3054
8,True Lies,0.2586
9,Avatar Purusha,0.2482


In [39]:
# Optional API helper for the final weighted field recommender.
def get_weighted_recommendations(query=SAMPLE_QUERY, top_n=TOP_N):
    matched_title, match_score, suggestions = find_best_title(query, movie_titles)
    if matched_title is None:
        return {"match": None, "score": match_score, "suggestions": suggestions, "recommendations": []}

    index = movies.index[movies["movie title"] == matched_title][0]
    recs = recommend_from_index(index, weighted_nn, weighted_features, movies, top_n=top_n)
    return {
        "match": matched_title,
        "score": match_score,
        "suggestions": suggestions,
        "recommendations": recs,
    }

get_weighted_recommendations(SAMPLE_QUERY)

{'match': 'Avatar',
 'score': 1.0,
 'suggestions': ['Avatar', 'Daata', 'Rafathar'],
 'recommendations': [{'movie title': 'Avatar 5', 'score': 0.8225},
  {'movie title': 'Avatar 4', 'score': 0.761},
  {'movie title': 'Avatar 3', 'score': 0.6746},
  {'movie title': 'Avatar: The Way of Water', 'score': 0.6233},
  {'movie title': 'The Abyss', 'score': 0.5231},
  {'movie title': 'Terminator 2: Judgment Day', 'score': 0.4013},
  {'movie title': 'Aliens', 'score': 0.3631},
  {'movie title': 'The Terminator', 'score': 0.3054},
  {'movie title': 'True Lies', 'score': 0.2586},
  {'movie title': 'Avatar Purusha', 'score': 0.2482}]}

In [ ]:
saved_files = sorted(str(path.name) for path in OUTPUT_DIR.iterdir())
saved_files

['fasttext_metadata.json',
 'fasttext_recommender.joblib',
 'fasttext_vectors.npy',
 'tfidf_features.npz',
 'tfidf_metadata.json',
 'tfidf_recommender.joblib',
 'word2vec_metadata.json',
 'word2vec_recommender.joblib',
 'word2vec_vectors.npy']

In [ ]:
print("Notebook implementation is organized into TF-IDF, Word2Vec, and fastText sections.")
print("Each section saves its own artifacts under the artifacts/ folder.")
print("Use get_section_recommendations('tfidf' | 'word2vec' | 'fasttext', query) for API-style reuse.")

Notebook implementation is organized into TF-IDF, Word2Vec, and fastText sections.
Each section saves its own artifacts under the artifacts/ folder.
Use get_section_recommendations('tfidf' | 'word2vec' | 'fasttext', query) for API-style reuse.
